In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

transform = transforms.Compose([
    transforms.Resize((28,28)),
    transforms.ToTensor(),  # Convert to tensor
])

# Load CIFAR-10 dataset
cifar_data = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transforms.ToTensor()  # Converts to tensor and normalizes to [0, 1]
)

print(f"CIFAR-10 loaded: {len(cifar_data)} images")
print(f"Image shape: {cifar_data[0][0].shape}")  # Should be (3, 32, 32)

In [ ]:
cifar_data[0]

In [ ]:
class ColorizationDataset(Dataset):
    """
    A dataset for image colorization.
    Returns (grayscale_image, color_image) pairs.

    Args:
        cifar_dataset: The CIFAR-10 dataset (already transformed to tensors)
    """

    def __init__(self, cifar_dataset):
        # TO DO: Store the dataset
        self.cifar_dataset = cifar_dataset
        self.toTensor = transforms.ToTensor()
        self.grayScale = transforms.Grayscale()

    def __len__(self):
        # TO DO: Return the number of samples
        return len(self.cifar_dataset)

    def rgb_to_grayscale(self, img):
        """
        Convert an RGB image to grayscale.

        Args:
            img: Tensor of shape (3, H, W) with values in [0, 1]

        Returns:
            Tensor of shape (1, H, W) with values in [0, 1]
        """
        # TO DO: Implement RGB to grayscale conversion
        # Hint: Gray = 0.299 * R + 0.587 * G + 0.114 * B
        imgGray = img[0] * 0.299 + img[1] * 0.587 + img[2] * 0.114
        return imgGray

    def __getitem__(self, idx):
        """
        Returns:
            grayscale_image: Tensor of shape (1, H, W)
            color_image: Tensor of shape (3, H, W)
        """
        # TO DO: Get the color image and convert to grayscale
        # Return (grayscale_image, color_image)
        img = self.cifar_dataset[idx][0]
        img_gray = self.rgb_to_grayscale(img).unsqueeze(0)

        return (img_gray,img)

In [ ]:
# Test your implementation
colorization_dataset = ColorizationDataset(cifar_data)

# Get a sample
gray_img, color_img = colorization_dataset[0]

print(f"Grayscale image shape: {gray_img.shape}")  # Should be (1, 32, 32)
print(f"Color image shape: {color_img.shape}")      # Should be (3, 32, 32)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(gray_img.squeeze(), cmap='gray')
axes[0].set_title('Grayscale (Input)')
axes[0].axis('off')
axes[1].imshow(color_img.permute(1, 2, 0))
axes[1].set_title('Color (Target)')
axes[1].axis('off')
plt.show()

In [ ]:
# Test with DataLoader
dataloader = DataLoader(colorization_dataset, batch_size=8, shuffle=True)

gray_batch, color_batch = next(iter(dataloader))
print(f"Batch grayscale shape: {gray_batch.shape}")  # Should be (8, 1, 32, 32)
print(f"Batch color shape: {color_batch.shape}")      # Should be (8, 3, 32, 32)

In [ ]:
## After this OverEnginering from mine
# I think the task reqiuerd train model to colorizartion but after I train, I read the task again and I found the model is not in the task
# So After this does not for the task
#
#
#
#
#
#
#

In [ ]:
from torch.utils.data import DataLoader, random_split

# 3. Split into Train and Test (e.g., 80/20 split)
train_size = int(0.8 * len(colorization_dataset))
test_size = len(colorization_dataset) - train_size

train_dataset, test_dataset = random_split(colorization_dataset, [train_size, test_size])

In [ ]:
train_loader = DataLoader(colorization_dataset, batch_size=8, shuffle=True)
test_loader = DataLoader(colorization_dataset, batch_size=8, shuffle=True)

In [ ]:
import math

def calK(ni,no,k,s,p):
  print(int((ni+2*p-k)/s)+1)


In [ ]:
calK(16, 32, 3, 2, 1)

In [ ]:
import torch.nn as nn

# Define Autoencoder Model
class CNNAutoEncoder(nn.Module):
    def __init__(self, encoding_dim=8, dropout_rate=0.2):
        super(CNNAutoEncoder, self).__init__()

        # Encoder (Feature Extraction)
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, 3, 2, 1),  # (28x28) → (14x14)
            nn.BatchNorm2d(16),
            nn.LeakyReLU(),
            nn.Dropout2d(dropout_rate),
            nn.Conv2d(16, 32, 3, 2, 1), # (14x14) → (7x7)
            nn.BatchNorm2d(32),
            nn.LeakyReLU(),
            nn.Dropout2d(dropout_rate),
            nn.Flatten(),  # Flatten to vector
            nn.Linear(2048, encoding_dim),
            nn.BatchNorm1d(encoding_dim),
            nn.Tanh(),
            nn.Dropout(dropout_rate)
        )

        # Decoder (Reconstruction)
        self.decoder = nn.Sequential(
            nn.Linear(encoding_dim, 2048),
            nn.BatchNorm1d(2048),
            nn.LeakyReLU(),
            nn.Dropout(dropout_rate),
            nn.Unflatten(1, (32, 8, 8)),  # Reshape back
            nn.ConvTranspose2d(32, 16, 3, 2, 1, output_padding=1),  # (7x7) → (14x14)
            nn.BatchNorm2d(16),
            nn.LeakyReLU(),
            nn.Dropout2d(dropout_rate),
            nn.ConvTranspose2d(16, 3, 3, 2, 1, output_padding=1),  # (14x14) → (28x28)
            nn.Sigmoid(),  # Normalize pixel values
        )

    def forward(self, x):
        encoder_out = self.encoder(x)
        decoder_out = self.decoder(encoder_out)
        return encoder_out, decoder_out

In [ ]:
from tqdm import tqdm    # Shows progress bar

# 🔹 Training Loop
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()  # Set model to training mode
    total_loss = 0

    for images, _ in tqdm(dataloader):  # Ignore labels since Autoencoders don't use them
        images = images.to(device)

        _, reconstructions = model(images)  # Forward pass (encoder + decoder)
        loss = criterion(reconstructions, images)  # Compute reconstruction loss

        optimizer.zero_grad()  # Reset gradients
        loss.backward()  # Backpropagation
        optimizer.step()  # Update weights

        total_loss += loss.item()

    avg_loss = total_loss / len(dataloader)
    return avg_loss  # No accuracy since it's not classification


In [ ]:
import torch.optim as optim


# Initialize the model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CNNAutoEncoder(encoding_dim=8).to(device)

# Print model summary
print(model)

# Define loss function and optimizer
criterion = nn.MSELoss()  # Measure reconstruction quality
optimizer = optim.AdamW(model.parameters(), lr=1e-4)  # AdamW optimizer
num_epochs = 5 # Number of epochs

# Store losses for plotting
train_losses = []

# Training loop
for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    train_losses.append(train_loss)

    print(f"Epoch {epoch+1}/{num_epochs}, Loss = {train_loss:.4f}")

In [ ]:
import matplotlib.pyplot as plt

# Plot loss curve
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(range(1, num_epochs+1), train_losses, label="Train Loss", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Loss Curve")
plt.legend()

plt.show()

In [ ]:
import matplotlib.pyplot as plt

# 🔹 Function to Display Original vs. Reconstructed Images
def show_original_vs_colorization(model, dataloader, device, num_images=3):
    model.eval()  # Set to evaluation mode

    # Get a batch of images
    images, _ = next(iter(dataloader))
    images = images[:num_images].to(device)  # Select 'num_images' samples

    # Get reconstructed images
    with torch.no_grad():
        _, colorization = model(images)
    colorization = colorization.cpu()

    # Plot original vs reconstructed images
    fig, axes = plt.subplots(2, num_images, figsize=(num_images * 2, 4))

    for i in range(num_images):
        # Original images (Top row)
        axes[0, i].imshow(images[i].cpu().squeeze(), cmap="gray")
        axes[0, i].axis("off")

        # Reconstructed images (Bottom row)
        axes[1, i].imshow(colorization[i].permute((1,2,0)), cmap="gray")
        axes[1, i].axis("off")

    axes[0, 0].set_title("Original Images", fontsize=12)
    axes[1, 0].set_title("Coloriziation Images", fontsize=12)
    plt.show()

# 🔹 Display Results
show_original_vs_colorization(model, train_loader, device)
